In [2]:
!pip install PyAutoGUI -q

import pyautogui


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import time
import pyautogui
time.sleep(2.0)

distance = 500
while distance > 0:
        pyautogui.drag(distance, 0, button='left', duration=0.5)   # move right
        distance -= 50
        pyautogui.drag(0, distance, button='left', duration=0.5)   # move down
        pyautogui.drag(-distance, 0, button='left', duration=0.5)  # move left
        distance -= 50
        pyautogui.drag(0, -distance, button='left', duration=0.5)  # move up

In [9]:
time.sleep(0.5)

hotkey = 'command' if 'mac' in pyautogui.platform.platform() else 'ctrl'

pyautogui.hotkey(hotkey, 't')

time.sleep(1.0)
pyautogui.typewrite('https://pyautogui.readthedocs.io', 0.01)

time.sleep(0.1)
pyautogui.hotkey(hotkey, 'l')

pyautogui.press('enter')

#### Application: Playing Online Games Using Faces

In [18]:
import cv2
import numpy as np
import pyautogui as gui
import time

gui.PAUSE = 0

model_path = 'data/models/res10_300x300_ssd_iter_140000.caffemodel'
prototxt_path = 'data/models/deploy.prototxt'

In [19]:
def play(prototxt_path, model_path):
    cap = cv2.VideoCapture(0)

    frame_width, frame_height = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    left_x, top_y = frame_width // 2 - 150, frame_height // 2 - 200
    right_x, bottom_y = frame_width // 2 + 150, frame_height // 2 + 200
    bbox = [left_x, right_x, bottom_y, top_y]

    while not cap.isOpened():
        cap = cv2.VideoCapture(0)

    while True:
        ret, frame = cap.read()
        if not ret:
            return 0
        
        frame = cv2.flip(frame, 1)

        frame = cv2.rectangle(frame, (left_x, top_y), (right_x, bottom_y), (0, 0, 255), 5)

        k = cv2.waitKey(10)
        if k == 27:
            return

In [20]:
net = cv2.dnn.readNetFromCaffe(prototxt_path, model_path)

In [21]:
def detect(net, frame):
    detected_faces = []
    (h, w) = frame.shape[:2]
    blob = cv2.dnn.blobFromImage(cv2.resize(frame, (300, 300)), 1.0, (300, 300), (104.0, 177.0, 123.0))
    net.setInput(blob)
    detections = net.forward()
    for i in range(0, detections.shape[2]):
        confidence = detections[0,0,i,2]
        if confidence > 0.5:
            box = detections[0,0,i,3:7] * np.array([w,h,w,h])
            (startX, startY, endX, endY) = box.astype("int")
            detected_faces.append({
                'start': (startX, startY),
                'end': (endX, endY),
                'confidence': confidence
            })

    return detected_faces

In [22]:
def drawFace(frame, detected_faces):
    for face in detected_faces:
        cv2.rectangle(frame, face['start'], face['end'], (0, 255, 0), 10)

    return frame

In [23]:
def checkRect(detected_faces, bbox):
    for face in detected_faces:
        x1, y1 = face['start']
        x2, y2 = face['end']
        if x1 > bbox[0] and x2 < bbox[1]:
            if y1 > bbox[3] and y2 < bbox[2]:
                return True
    return False

In [24]:
def move(detected_faces, bbox):
    global last_mov
    for face in detected_faces:
        x1, y1 = face['start']
        x2, y2 = face['end']

        if checkRect(detected_faces, bbox):
            last_mov = 'center'
            return
        
        elif last_mov == 'center':
            if x1 < bbox[0]:
                gui.press('left')
                last_mov = 'left'
            elif x2 > bbox[1]:
                gui.press('right')
                last_mov = 'right'

            if y2 > bbox[2]:
                gui.press('down')
                last_mov = 'down'
            elif y1 < bbox[3]:
                gui.press('up')
                last_mov = 'up'

            if last_mov != 'center':
                print(last_mov)

In [25]:
def play(prototxt_path, model_path):
    global last_mov
    prev_frame_time = 0
    new_frame_time = 0

    net = cv2.dnn.readNetFromCaffe(prototxt_path, model_path)
    cap = cv2.VideoCapture(0)

    # Counter for skipping frame.
    count = 0

    # Used to initialize the game.
    init = 0

    frame_width, frame_height = int(cap.get(3)), int(cap.get(4))

    left_x, top_y = frame_width // 2 - 150, frame_height // 2 - 200
    right_x, bottom_y = frame_width // 2 + 150, frame_height // 2 + 200
    bbox = [left_x, right_x, bottom_y, top_y]

    while not cap.isOpened():
        cap = cv2.VideoCapture(0)

    while True:
        fps = 0
        ret, frame = cap.read()

        if not ret:
            return 0

        frame = cv2.flip(frame, 1)
        detected_faces = detect(net, frame)
        frame = drawFace(frame, detected_faces)
        frame = cv2.rectangle(
            frame, (left_x, top_y), (right_x, bottom_y), (0, 0, 255), 5)

        if count % 2 == 0:
            if init == 0:
                if checkRect(detected_faces, bbox):
                    init = 1
                    cv2.putText(
                        frame, 'Game is running', (100, 100),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
                    cv2.waitKey(10)
                    last_mov = 'center'
                    gui.click(x=500, y=500)
            else:

                move(detected_faces, bbox)
                cv2.waitKey(50)
        new_frame_time = time.time()
        fps = int(1 / (new_frame_time - prev_frame_time))
        prev_frame_time = new_frame_time

        frame = cv2.putText(
            frame, str(fps) + 'FPS', (200, 100),
            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 0), 2)
        cv2.imshow('camera_feed', frame)
        count += 1

        k = cv2.waitKey(5)
        if k == 27:
            return

In [ ]:
last_mov = ''
play(prototxt_path, model_path)

left
right
left
down
left
right
right
left
up
left
down
up
right
left
right
up
down
up
right
right
right
left
up
right
right
right
down
right
right
up
left
down
left
right
right
left
down
right
right
up
down
right
right
down
up
left
down
up
right
left
right
down


KeyboardInterrupt: 

: 